# Creator Bandit Simulation: A Year of Adaptive Content Publishing

**Goal:** Simulate 52 weeks of content publishing using the Creator Bandit Playbook. See how adaptive allocation outperforms random publishing and handles arm retirement.

**What you'll learn:**
- How bandits tilt toward high-engagement content while staying curious
- The impact of quarterly arm retirement on strategy evolution
- Why optimizing for read ratio (not views) prevents clickbait training

**Time:** 15 minutes

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("Ready to simulate creator bandits!")

In [ ]:
learning_objectives(["**Exploration budget:** Change `posts_per_week` allocation (2 to top vs 3 to top)", "**Retirement frequency:** Change `[12, 24, 36]` to `[8, 16, 24, 32, 40]` (monthly)", "**Arm count:** Start with 8 arms instead of 6", "**Noise level:** Change `noise=0.12` in `publish_content()` to 0.20 (harder to learn)"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Define Arms: Topic × Format Combinations

We're a commodity trading newsletter with:
- **3 topics:** Market Analysis, Trading Psychology, Risk Management
- **2 formats:** Long-form Essays, Tweet Threads
- **6 arms total**

Each arm has a true "read ratio" (% of readers who finish the content).

In [ ]:
# Define 6 content arms
arms = [
    "Market Analysis × Essay",
    "Market Analysis × Thread",
    "Trading Psychology × Essay",
    "Trading Psychology × Thread",
    "Risk Management × Essay",
    "Risk Management × Thread"
]

# True read ratios (unknown to creator)
true_read_ratios = {
    arms[0]: 0.52,  # Market × Essay: best
    arms[1]: 0.38,  # Market × Thread
    arms[2]: 0.48,  # Psychology × Essay
    arms[3]: 0.35,  # Psychology × Thread
    arms[4]: 0.28,  # Risk × Essay: worst
    arms[5]: 0.42   # Risk × Thread
}

def publish_content(arm, noise=0.12):
    """Simulate publishing: return read ratio with noise"""
    true_ratio = true_read_ratios[arm]
    observed = np.random.normal(true_ratio, noise)
    return np.clip(observed, 0, 1)

print("Arms defined:")
for arm, ratio in true_read_ratios.items():
    print(f"  {arm}: {ratio:.0%} read ratio (true)")

## Implement the Creator Bandit Playbook

**Phase 1 (Weeks 1-3):** Publish evenly across all 6 arms

**Phase 2 (Weeks 4-52):** 
- Top 2 arms get 60% of posts
- Remaining arms get 40% (exploration budget)

**Phase 3 (Weeks 12, 24, 36):** Retire worst arm, add new one

In [ ]:
class CreatorBandit:
    def __init__(self, arms, posts_per_week=5):
        self.arms = arms[:]
        self.posts_per_week = posts_per_week
        self.n_pulls = {arm: 0 for arm in arms}
        self.rewards = {arm: [] for arm in arms}
        self.history = []
        self.retired = []
        
    def get_mean(self, arm, window=None):
        if not self.rewards[arm]:
            return 0.0
        data = self.rewards[arm]
        if window:
            data = data[-window:]
        return np.mean(data)
    
    def select_arms_for_week(self, week):
        # Phase 1: Even exploration (weeks 0-2)
        if week < 3:
            return np.random.choice(
                self.arms, 
                size=self.posts_per_week, 
                replace=True
            )
        
        # Phase 2: Tilted exploitation
        means = {arm: self.get_mean(arm) 
                 for arm in self.arms}
        ranked = sorted(self.arms, 
                       key=lambda a: means[a], 
                       reverse=True)
        
        # Top 2 get 3 posts, others get 2 posts total
        week_arms = (
            [ranked[0]] * 2 + 
            [ranked[1]] * 1 + 
            [ranked[i] for i in range(2, len(ranked))][:2]
        )
        return week_arms[:self.posts_per_week]
    
    def retire_worst(self):
        if len(self.arms) <= 3:
            return None
        
        # Find worst with enough pulls
        eligible = [a for a in self.arms 
                   if self.n_pulls[a] >= 10]
        if not eligible:
            return None
            
        means = {a: self.get_mean(a, window=20) 
                for a in eligible}
        worst = min(eligible, key=lambda a: means[a])
        
        self.arms.remove(worst)
        self.retired.append(worst)
        return worst
    
    def add_arm(self, new_arm, true_ratio):
        self.arms.append(new_arm)
        self.n_pulls[new_arm] = 0
        self.rewards[new_arm] = []
        true_read_ratios[new_arm] = true_ratio

print("Creator Bandit class ready!")

## Run 52-Week Simulation

Watch the bandit learn which content types work best and adapt its strategy.

In [ ]:
# Initialize bandit
bandit = CreatorBandit(arms, posts_per_week=5)

# Track performance
weekly_engagement = []
arm_pulls_over_time = defaultdict(list)

# Simulate 52 weeks
for week in range(52):
    # Select arms for this week
    week_arms = bandit.select_arms_for_week(week)
    
    # Publish content
    week_total_engagement = 0
    for arm in week_arms:
        reward = publish_content(arm)
        bandit.n_pulls[arm] += 1
        bandit.rewards[arm].append(reward)
        bandit.history.append((week, arm, reward))
        week_total_engagement += reward
    
    weekly_engagement.append(
        week_total_engagement / len(week_arms)
    )
    
    # Track arm usage
    for arm in arms + [a for a in bandit.arms 
                       if a not in arms]:
        if arm in bandit.arms:
            arm_pulls_over_time[arm].append(
                bandit.n_pulls.get(arm, 0)
            )
    
    # Quarterly arm retirement
    if week in [12, 24, 36]:
        retired = bandit.retire_worst()
        if retired:
            print(f"Week {week}: Retired '{retired}'")
            # Add new arm
            new_name = f"Podcast {week}"
            new_ratio = np.random.uniform(0.35, 0.55)
            bandit.add_arm(new_name, new_ratio)
            print(f"Week {week}: Added '{new_name}' "
                  f"(true ratio: {new_ratio:.0%})")

print(f"\nSimulation complete! Published {len(bandit.history)} posts.")

## Visualize Arm Selection Over Time

See how the bandit shifted publishing toward top performers.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Cumulative pulls by arm
for arm in arms:
    if arm in arm_pulls_over_time:
        weeks = range(len(arm_pulls_over_time[arm]))
        ax1.plot(weeks, arm_pulls_over_time[arm], 
                label=arm, linewidth=2)

ax1.set_xlabel('Week')
ax1.set_ylabel('Cumulative Posts Published')
ax1.set_title('Arm Selection Over Time (Bandit Adapts)')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# Plot 2: Weekly average engagement
ax2.plot(weekly_engagement, linewidth=2, color='green')
ax2.axhline(y=np.mean(list(true_read_ratios.values())), 
           color='red', linestyle='--', 
           label='Random publishing baseline')
ax2.set_xlabel('Week')
ax2.set_ylabel('Average Read Ratio')
ax2.set_title('Weekly Engagement (Bandits Learn & Improve)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal avg engagement: {np.mean(weekly_engagement[12:]):.1%}")
print(f"Random baseline: {np.mean(list(true_read_ratios.values())):.1%}")

## Compare: Bandit vs Random Publishing

What if you just published randomly instead of using bandits?

In [ ]:
# Simulate random publishing (baseline)
random_engagement = []
for week in range(52):
    week_arms = np.random.choice(arms, size=5)
    week_rewards = [publish_content(a) for a in week_arms]
    random_engagement.append(np.mean(week_rewards))

# Calculate lift
bandit_total = sum(weekly_engagement)
random_total = sum(random_engagement)
lift = (bandit_total - random_total) / random_total

plt.figure(figsize=(12, 5))
plt.plot(weekly_engagement, label='Bandit Strategy', 
        linewidth=2, color='green')
plt.plot(random_engagement, label='Random Publishing', 
        linewidth=2, color='gray', alpha=0.6)
plt.xlabel('Week')
plt.ylabel('Average Read Ratio')
plt.title(f'Bandit vs Random: {lift:.1%} Engagement Lift')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nBandit total engagement: {bandit_total:.1f}")
print(f"Random total engagement: {random_total:.1f}")
print(f"Improvement: {lift:.1%}")

## Final Performance Report

Which arms survived? Which were retired?

In [ ]:
print("=" * 60)
print("FINAL PERFORMANCE REPORT (52 weeks)")
print("=" * 60)

print("\nACTIVE ARMS:")
for arm in bandit.arms:
    pulls = bandit.n_pulls.get(arm, 0)
    if pulls > 0:
        mean = bandit.get_mean(arm)
        true = true_read_ratios[arm]
        print(f"  {arm}")
        print(f"    Posts: {pulls}, "
              f"Observed: {mean:.1%}, True: {true:.1%}")

print("\nRETIRED ARMS:")
for arm in bandit.retired:
    pulls = bandit.n_pulls.get(arm, 0)
    if pulls > 0:
        mean = bandit.get_mean(arm)
        true = true_read_ratios[arm]
        print(f"  {arm}")
        print(f"    Posts: {pulls}, "
              f"Observed: {mean:.1%}, True: {true:.1%}")

print("\n" + "=" * 60)

## Modify This: Experiment with Parameters

Try changing these and re-run the simulation:

1. **Exploration budget:** Change `posts_per_week` allocation (2 to top vs 3 to top)
2. **Retirement frequency:** Change `[12, 24, 36]` to `[8, 16, 24, 32, 40]` (monthly)
3. **Arm count:** Start with 8 arms instead of 6
4. **Noise level:** Change `noise=0.12` in `publish_content()` to 0.20 (harder to learn)

**Questions to explore:**
- Does more frequent retirement help or hurt?
- What if all arms have similar performance (hard problem)?
- How much exploration budget is optimal?

In [ ]:
# Your experiments here
# Copy the simulation code above and modify parameters

## Summary

**What we learned:**
1. Bandits automatically tilt toward high-engagement content (top 2 arms get 60% of posts by week 12)
2. Quarterly arm retirement creates an "evolutionary" system — strategies adapt over time
3. Typical lift vs random publishing: **15-25%** more engagement with the same effort
4. The key is using **quality metrics** (read ratio) not vanity metrics (views)

**For commodity traders:**
- Same framework applies to research report formats, alert channels, client communication
- Optimize for "trader took action" not "trader opened email"
- Retire underperforming formats quarterly to stay relevant

**Next steps:**
- Notebook 02: Apply bandits to conversion rate optimization
- Notebook 03: Gallery of business applications (pricing, onboarding, email testing)

In [ ]:
key_takeaways(["1. Bandits automatically tilt toward high-engagement content (top 2 arms get 60%", "more engagement with the same effort", "(read ratio) not vanity metrics (views)", "- Same framework applies to research report formats, alert channels, client comm", "Optimize for "trader took action" not "trader opened email"", "Retire underperforming formats quarterly to stay relevant"])